# XGBoost (BOOKING only) — v1

- BOOKING 만 학습
- feature 30개 고정 + 순서 고정
- 산출물: `model.joblib`, `meta.json`, `input_features.json`


In [ ]:
import json
import time
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

args = SimpleNamespace(
    RUN_VERSION='xgboost_booking_v1',
    DATA_ROOTS=[],
    MODEL_SAVE_DIR=Path('../model_joblib') / 'xgboost_booking_v1' / 'booking',
    RANDOM_STATE=42,
    TEST_SIZE=0.2,
    SAVE_MODEL=True,
    # XGBoost params
    N_ESTIMATORS=2500,
    LEARNING_RATE=0.03,
    MAX_DEPTH=5,
    MIN_CHILD_WEIGHT=1.0,
    SUBSAMPLE=0.9,
    COLSAMPLE_BYTREE=0.9,
    REG_ALPHA=0.0,
    REG_LAMBDA=1.0,
    GAMMA=0.0,
    EARLY_STOPPING_ROUNDS=100,
)
args


In [ ]:
def _expand_inputs(inputs: list[str]) -> list[Path]:
    out: list[Path] = []
    for raw in inputs:
        raw = str(raw).strip().strip('"').strip("'")
        if not raw:
            continue
        if any(ch in raw for ch in ['*', '?', '[']):
            out.extend(sorted(Path().glob(raw)))
        else:
            out.append(Path(raw))
    seen = set()
    uniq: list[Path] = []
    for p in out:
        rp = str(p.resolve()) if p.exists() else str(p)
        if rp in seen:
            continue
        seen.add(rp)
        uniq.append(p)
    return uniq

# Default roots (edit as needed)
default_roots = [
    r'C:/Users/SSAFY/Desktop/ws/free/develop-ai/S14P31A203/services/ai/data/train_data_0518/data_0515_1400_gt_v2',
    r'C:/Users/SSAFY/Desktop/ws/free/develop-ai/S14P31A203/services/ai/data/train_data_0518/data_tickle_0515_human_v2',
]

args.DATA_ROOTS = list(default_roots)
roots = _expand_inputs(args.DATA_ROOTS)
print('n_roots =', len(roots))
for p in roots[:10]:
    print(' -', p)
assert len(roots) > 0, 'No dataset roots found'


In [ ]:
FEATURES = [
  'reclick_rate',
  'misclick_rate',
  'double_click_rate',
  'mouse_overshoot_flag',
  'mousemove_event_rate',
  'pre_click_scroll_flag',
  'time_to_first_click_ms',
  'mouse_acceleration_mean',
  'mouse_speed_change_mean',
  'pre_click_hover_time_ms',
  'mouse_stop_segment_count',
  'mouse_avg_speed_px_per_ms',
  'mouse_hover_dwell_time_ms',
  'mouse_max_speed_px_per_ms',
  'mouse_path_curvature_mean',
  'pre_click_mousemove_count',
  'click_position_repeat_rate',
  'mouse_direction_change_count',
  'mouse_path_straightness_score',
  'edge_or_fixed_point_visit_rate',
  'mouse_total_travel_distance_px',
  'click_sequence_consistency_score',
  'immediate_post_render_click_rate',
  'pre_click_path_300ms_straightness',
  'pre_click_path_500ms_straightness',
  'click_offset_from_element_center_px',
  'time_from_element_visible_to_click_ms',
  'pre_click_path_300ms_total_distance_px',
  'pre_click_path_500ms_total_distance_px',
  'time_from_element_clickable_to_click_ms',
]
print('n_features =', len(FEATURES))


In [ ]:
def _safe_lower(x) -> str:
    return str(x).strip().lower()

def _infer_type_and_label(path: Path):
    # Expect .../<type>/<allow|block>/<file>.json
    try:
        label_dir = _safe_lower(path.parent.name)
        type_dir = str(path.parent.parent.name).strip()
        if label_dir not in ('allow', 'block'):
            return None, None
        if not type_dir:
            return None, None
        return type_dir, label_dir.upper()
    except Exception:
        return None, None

def _load_trial_json(path: Path) -> dict:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_from_roots(roots: list[Path]) -> pd.DataFrame:
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob('*.json'):
            type_dir, label = _infer_type_and_label(p)
            if type_dir is None:
                continue
            obj = _load_trial_json(p)
            feats = None
            if isinstance(obj, dict):
                feats = obj.get('features')
                if feats is None:
                    feats = obj.get('metrics')
                if feats is None:
                    feats = {k: v for k, v in obj.items() if k not in ('trialID', 'trialId', 'type', 'label', 'summary', 'metrics', 'features')}
            if feats is None:
                feats = {}
            rows.append({
                'source_root': str(root),
                'source_file': str(p),
                'trialID': (obj.get('trialID') if isinstance(obj, dict) else None) or (obj.get('trialId') if isinstance(obj, dict) else None),
                'type': type_dir,
                'label': label,
                **(feats or {}),
            })
    return pd.DataFrame(rows)

df_all = load_from_roots(roots)
print('df_all.shape =', df_all.shape)
df_all[['type', 'label', 'source_file']].head()


In [ ]:
# BOOKING only
df = df_all.copy()
df['type'] = df['type'].astype(str).fillna('')
df = df[df['type'].str.lower() == 'booking'].copy()
print('df(booking).shape =', df.shape)
print(df['label'].value_counts(dropna=False))


In [ ]:
# If not installed: pip install xgboost
from xgboost import XGBClassifier

def build_xy(sub: pd.DataFrame):
    sub = sub.copy()
    for c in FEATURES:
        if c not in sub.columns:
            sub[c] = np.nan

    y = (sub['label'].astype(str).str.upper() == 'BLOCK').astype(int)
    X = sub[FEATURES].apply(pd.to_numeric, errors='coerce')

    med = X.median(numeric_only=True)
    X = X.fillna(med).fillna(0)
    return X, y, med

def train_booking(sub: pd.DataFrame) -> dict:
    X, y, med = build_xy(sub)
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=args.TEST_SIZE,
        random_state=args.RANDOM_STATE,
        stratify=y,
    )

    pos = int(y_train.sum())
    neg = int(len(y_train) - pos)
    scale_pos_weight = float(neg / max(pos, 1))
    print('train pos/neg =', pos, '/', neg, 'scale_pos_weight =', scale_pos_weight)

    clf = XGBClassifier(
        n_estimators=args.N_ESTIMATORS,
        learning_rate=args.LEARNING_RATE,
        max_depth=args.MAX_DEPTH,
        min_child_weight=args.MIN_CHILD_WEIGHT,
        subsample=args.SUBSAMPLE,
        colsample_bytree=args.COLSAMPLE_BYTREE,
        reg_alpha=args.REG_ALPHA,
        reg_lambda=args.REG_LAMBDA,
        gamma=args.GAMMA,
        objective='binary:logistic',
        eval_metric=['logloss', 'auc'],
        scale_pos_weight=scale_pos_weight,
        random_state=args.RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist',
    )

    import inspect

    start = time.time()
    fit_sig = None
    try:
        fit_sig = inspect.signature(clf.fit)
    except Exception:
        fit_sig = None

    fit_kwargs = {
        'eval_set': [(X_valid, y_valid)],
        'verbose': False,
    }

    # xgboost version compatibility:
    # - some versions support `early_stopping_rounds`
    # - some require callbacks
    if fit_sig is not None and 'early_stopping_rounds' in fit_sig.parameters:
        fit_kwargs['early_stopping_rounds'] = args.EARLY_STOPPING_ROUNDS
        clf.fit(X_train, y_train, **fit_kwargs)
    else:
        # Try callback-based early stopping (newer xgboost)
        try:
            from xgboost.callback import EarlyStopping
            fit_kwargs['callbacks'] = [EarlyStopping(rounds=args.EARLY_STOPPING_ROUNDS, save_best=True)]
            clf.fit(X_train, y_train, **fit_kwargs)
        except Exception:
            # Fallback: no early stopping
            clf.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

    elapsed = time.time() - start

    proba_valid = clf.predict_proba(X_valid)[:, 1]
    pred_valid = (proba_valid >= 0.5).astype(int)

    auc = roc_auc_score(y_valid, proba_valid)
    ap = average_precision_score(y_valid, proba_valid)
    ll = log_loss(y_valid, proba_valid)
    acc = accuracy_score(y_valid, pred_valid)
    p, r, f1, _ = precision_recall_fscore_support(y_valid, pred_valid, average='binary', zero_division=0)

    return {
        'type': 'booking',
        'n': int(len(y)),
        'pos': int(y.sum()),
        'pos_rate': float(y.mean()),
        'n_features_used': int(X.shape[1]),
        'best_iteration': int(getattr(clf, 'best_iteration', 0) or 0),
        'elapsed_sec': float(elapsed),
        'AUC': float(auc),
        'AP': float(ap),
        'LogLoss': float(ll),
        'Acc': float(acc),
        'Precision': float(p),
        'Recall': float(r),
        'F1': float(f1),
        'model': clf,
        'features_used': list(X.columns),
        'median_imputer': med,
        'valid_pred': pd.DataFrame({'type': 'booking', 'y_true': y_valid.values, 'y_pred': pred_valid, 'proba': proba_valid}),
        'cm': confusion_matrix(y_valid, pred_valid),
        'report': classification_report(y_valid, pred_valid, digits=4),
    }

res = train_booking(df)
pd.DataFrame([{k: v for k, v in res.items() if k not in ['model','valid_pred','cm','report','median_imputer','features_used']}])


In [ ]:
# Valid proba distribution
vp = res['valid_pred'].copy()
plt.figure(figsize=(7.6, 3.6))
sns.histplot(data=vp, x='proba', hue=vp['y_true'].map({0:'ALLOW',1:'BLOCK'}), bins=30, stat='count', common_norm=False)
plt.title('booking proba (valid)')
plt.xlabel('P(BLOCK)')
plt.tight_layout()
plt.show()

print('confusion_matrix:\n', res['cm'])
print('\nclassification_report:\n', res['report'])


In [ ]:
if args.SAVE_MODEL:
    args.MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    def _json_safe(obj):
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, (list, tuple)):
            return [_json_safe(x) for x in obj]
        if isinstance(obj, dict):
            return {str(k): _json_safe(v) for k, v in obj.items()}
        return obj

    args_dump = _json_safe(vars(args))

    joblib.dump(res['model'], args.MODEL_SAVE_DIR / 'model.joblib')

    meta = {
        'feature_names': list(res['features_used']),
        'n_features': int(len(res['features_used'])),
        'label_mapping': {'human': 0, 'macro': 1},
        'type': 'BOOKING',
        'run_version': args.RUN_VERSION,
        'args': args_dump,
        'dataset_roots': [str(p) for p in roots],
    }
    (args.MODEL_SAVE_DIR / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    (args.MODEL_SAVE_DIR / 'input_features.json').write_text(
        json.dumps(list(res['features_used']), ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    res['valid_pred'].to_csv(args.MODEL_SAVE_DIR / 'valid_predictions.csv', index=False)
    pd.DataFrame({'cm': res['cm'].reshape(-1)}).to_csv(args.MODEL_SAVE_DIR / 'confusion_matrix_flat.csv', index=False)
    (args.MODEL_SAVE_DIR / 'classification_report.txt').write_text(res['report'], encoding='utf-8')

    print('saved to', args.MODEL_SAVE_DIR.resolve())
